# einops-reduce — faded example 2: Compute per-(batch, channel) spatial minimum with keepdim placeholders

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce`. Running the beacon reports progress on the `Einops: Reduce` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import reduce

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-reduce`**, which bridges to the bank subtopic `Einops: Reduce` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

Using `()` on the right side of a `reduce` pattern keeps the collapsed axis as a size-1 placeholder for broadcasting. `'b c h w -> b c () ()'` reduces the `h` and `w` axes but leaves them as size-1, producing shape `(B, C, 1, 1)` instead of `(B, C)`. This lets the result broadcast back against the original `(B, C, H, W)` tensor.

## Faded exercise 2

Complete `spatial_min_keepdim`. Use `reduce` with `()` placeholders so the output can broadcast back against `x`.

**Fill in:** The `reduce(x, 'b c h w -> b c () ()', 'min')` call that keeps size-1 spatial axes for broadcasting.

In [ ]:
from einops import reduce
import torch as t

def spatial_min_keepdim(x):
    """x: (B, C, H, W) -> (B, C, 1, 1)"""
    raise NotImplementedError()  # TODO: The `reduce(x, 'b c h w -> b c () ()', 'min')` call that keeps size-1 spatial axes for broadcasting.


def _test():
    import torch as t
    t.manual_seed(7)
    B, C, H, W = 2, 3, 5, 5
    x = t.randn(B, C, H, W)
    out = spatial_min_keepdim(x)
    assert out.shape == (B, C, 1, 1)
    # Must broadcast: x - spatial_min should have non-negative values
    zeroed = x - out
    assert (zeroed >= -1e-6).all()
    # Shape mismatch would raise an error in the above subtraction
    expected = x.amin(dim=(2, 3), keepdim=True)
    assert t.allclose(out, expected)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from einops import reduce
import torch as t

def spatial_min_keepdim(x):
    """x: (B, C, H, W) -> (B, C, 1, 1)"""
    return reduce(x, 'b c h w -> b c () ()', 'min')
```
</details>